轻量类型的约束，返回的是字典，类型不匹配的时候不抛出异常

## 是py3.8引入的一种类型提示工具，即是带有类型声明的字典结构，适合需要快速定义字典结构切无需Pydantic的重量级功能场景

In [9]:
from typing_extensions import TypedDict, Annotated
from typing import Literal, Optional, List
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)

class MovieDict(TypedDict):
    title:str
    year:int
    director:str
    rating:float
movie: MovieDict ={
    "title":"盗梦空间",
    "year":2010,
    "director":"克里斯托佛丶诺兰",
    "rating":8.8,

}


print(movie)


{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托佛丶诺兰', 'rating': 8.8}


Annotated 用来在类型之外，再附加一些额外的信息，即元数据，类似于Pydantic中的Field

In [11]:
from typing_extensions import TypedDict,Annotated
from typing import Literal, Optional
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)

class MovieDict(TypedDict):
    title:Annotated[str,"电影的名称"]
    year:Annotated[int,"上映年份"]
    director:Annotated[str,"导演姓名"]
    rating:Annotated[float,"电影评分"]
movie: MovieDict ={
    "title":"盗梦空间",
    "year":2010,
    "director":"克里斯托佛丶诺兰",
    "rating":8.8,
}
structed_model=model.with_structured_output(MovieDict)
message=[HumanMessage("给我介绍一下《唐伯虎点秋香》的信息")]
response=structed_model.invoke(message)
print(response)


{'title': '《唐伯虎点秋香》', 'year': 1993, 'director': '李力持', 'rating': 8.7}


嵌套结构的使用 以及 ...的使用，但是...的效果受模型供应商的影响  ...可以理解为必填项

In [16]:
from typing import Literal, Optional,List
class Actor(TypedDict):
    """
    演员的信息
    """
    name:Annotated[str,"演员姓名"]
    role:Annotated[str,"扮演的角色"]
class MovieTypeDict(TypedDict):
    """
    电影的信息
    """
    title:Annotated[str,"电影的名称"]
    year:Annotated[int,"上映年份"]
    director:Annotated[str,"导演姓名"]
    rating:Annotated[float,"电影评分"]
    cast:Annotated[List[Actor],"电影的演员列表"]
structed_model=model.with_structured_output(MovieTypeDict)
message=[HumanMessage("根据这段话抽取电影的信息，不包含的信息可以留空：盗梦空间在2018年上映，导演是克里斯托佛丶诺兰")]
response=structed_model.invoke(message)
print(response)


{'title': '盗梦空间', 'year': 2018, 'director': '克里斯托佛·诺兰'}
